## Complements

In [ ]:
import pandas as pd
%run ../utils/complements.py
%run ../utils/results.py

file_path_root = "../data/validation/complements/"

# Load files
product_df = pd.read_csv('../data/cleaned/product-info-full.csv')
orders_full_df = pd.read_csv('../dataset/order_products__prior.csv')

focus_products = [16696,24852,30696]

dept1_df = pd.read_csv('../data/validation/pairwise/pairwise-dept1.csv')
dept7_df = pd.read_csv('../data/validation/pairwise/pairwise-dept7.csv')
dept4_df = pd.read_csv('../data/validation/pairwise/pairwise-dept4.csv')

pairwise_df = pd.concat([dept1_df, dept4_df, dept7_df], ignore_index=True)

num_orders, min_pij = get_min_pij()

lift_df = compute_lift(focus_products, pairwise_df, min_pij=min_pij, total_orders=num_orders)
complements_df = compute_hybrid_score(lift_df, focus_products, top_n=10)
cii_df = compute_complement_impact_index(complements_df, pairwise_df)
network_df = compute_network_enhanced_impact(cii_df, pairwise_df)
network_df.to_csv(f"{file_path_root}minimal-complements.csv", index=False)

show_comp_results(network_df)

Computing lift: 100%|██████████| 3/3 [00:07<00:00,  2.48s/it]



Product: Coke Classic  (ID: 16696)


,comp_name,impact_index_j,aisle
0,Lemon Lime Soda Caffeine Free,1.000000,soft drinks
1,Classic Caffeine Free Soda,0.716798,soft drinks
2,Vanilla Coke,0.469591,soft drinks
3,Ginger Soda,0.379171,soft drinks
4,Chocolate Favorites Fun Size Variety Pack,0.305279,candy chocolate
5,Seasoned Black Cherry Barbecue Pork jerky,0.279181,popcorn jerky
6,Multi-Grain English Muffins,0.228437,breakfast bakery
7,Deluxe Mixed Nuts,0.199602,nuts seeds dried fruit
8,Roasted Garlic Hummus with Pretzels,0.186160,fresh dips tapenades
9,Miniatures Assortment Party Bag,0.174522,candy chocolate



Product: Banana  (ID: 24852)


,comp_name,impact_index_j,aisle
0,Disney Frozen Kids Yogurt,0.180028,yogurt
1,2nd Foods Organic Pear and Spinach Baby Food,0.166696,baby food formula
2,Shells & White Cheddar Mac & Cheese Family Siz...,0.161638,instant foods
3,"Veg and Fruit Puree, 100%, Organic, Sweet Pota...",0.159586,baby food formula
4,Eggs,0.158606,eggs
5,Cashew & Ginger Spice Fruit & Nut Bar,0.152944,energy granola bars
6,Humm! Cocktail Hummus Roasted Pine Nuts,0.150738,fresh dips tapenades
7,Reduced Fat Shredded Mozzarella Cheese,0.150446,packaged cheese
8,Trop50 Some Pulp Orange Juice,0.147687,refrigerated
9,Slim Cut Reduced Fat 2% Milk Sharp Cheddar Cheese,0.146544,packaged cheese



Product: Eggo Homestyle Waffles  (ID: 30696)


,comp_name,impact_index_j,aisle
0,Original Thin Sausage Pizza,0.563829,frozen pizza
1,Natural Butter Flavor Lite Maple Syrup,0.397202,honeys syrups nectars
2,Butter Rich Maple Syrup,0.366146,honeys syrups nectars
3,Original Patties (100965) 12 Oz Breakfast,0.290518,hot dogs bacon sausage
4,Mini Pancakes,0.289250,frozen breakfast
5,Original Lite Syrup,0.255276,honeys syrups nectars
6,Natural Low Moisture Part Skim Mozzarella Chee...,0.252871,packaged cheese
7,Original Chicken Breast Nuggets,0.241823,packaged poultry
8,Cinnamon French Toaster Sticks,0.237544,frozen breakfast
9,Mild Taco Seasoning Mix,0.226871,marinades meat preparation


In [ ]:
%run ../utils/complements.py

sampled_products = pd.read_csv('../data/validation/sampled-products.csv')
pairwise_df = pd.read_csv('../data/validation/sample-pairwise.csv')
product_df = pd.read_csv('../data/cleaned/product-info-full.csv')

num_orders, min_pij = get_min_pij()

focus_products = sampled_products['product_id'].to_list()

lift_df = compute_lift(focus_products, pairwise_df, min_pij=min_pij, total_orders=num_orders)
complements_df = compute_hybrid_score(lift_df, focus_products, top_n=10)
cii_df = compute_complement_impact_index(complements_df, pairwise_df)
network_df = compute_network_enhanced_impact(cii_df, pairwise_df)
network_df.to_csv(f"{file_path_root}sample-complements.csv", index=False)

total_impact = compute_total_impact(network_df, product_df)
total_impact.to_csv(f"{file_path_root}sample-total-impact.csv", index=False)

Computing lift: 100%|██████████| 5000/5000 [00:45<00:00, 109.47it/s]


In [1]:
%run ../utils/pairwise.py
%run ../utils/sampling.py
%run ../utils/complements.py
import pandas as pd

# Load files
all_orders_df = pd.read_csv('../data/cleaned/order-products-full.csv')
products_df = pd.read_csv('../data/cleaned/product-info-full.csv')

file_path_base = "../data/validation/complements/"

# Get list of sampled products and split orders into train and test sets
sampled_products, train_df, test_df = sample_products_and_split_orders(
    products_df,
    all_orders_df,
    target_sample_size=5000,
    min_orders=50,
    test_size=0.2,
    random_state=42
)

# Simplify train orders df
order_product_df = train_df[['order_id', 'product_id']]

# Compute probabilities for the train set
product_pair_df = compute_pairwise_probabilities_sample(order_product_df, 
    sampled_products,
    output_csv=f"{file_path_base}train-pairwise.csv",
    batch_size=1000
)

# Simplify test orders df
order_product_test_df = test_df[['order_id', 'product_id']]

# Compute probabilities for the test set
product_pair_test_df = compute_pairwise_probabilities_sample(order_product_test_df,
    sampled_products,
    output_csv=f"{file_path_base}test-pairwise.csv",
    batch_size=1000
)

Sampled 5000 products from 26686 eligible products.
Train orders: 5049729 rows, Test orders: 1264861 rows


100%|██████████| 5/5 [00:53<00:00, 10.64s/it]


Completed computation. Saved to ../data/validation/complements/train-pairwise.csv


100%|██████████| 5/5 [00:17<00:00,  3.56s/it]

Completed computation. Saved to ../data/validation/complements/test-pairwise.csv


In [34]:
%run ../utils/complements.py
import pandas as pd


# Load dataframes
pairwise_train_df = pd.read_csv(f"{file_path_base}train-pairwise.csv")
pairwise_test_df = pd.read_csv(f"{file_path_base}test-pairwise.csv")
orders_test_df = order_product_test_df.copy()

num_orders, min_pij = get_min_pij()

results = run_samples_example(
    pairwise_train_df,
    pairwise_test_df,
    sampled_products,
    orders_test_df,
    num_orders,
    min_pij,
    10
)


--- Sample 1/10 (size=100) ---


Computing lift: 100%|██████████| 100/100 [00:01<00:00, 78.21it/s]


Complement pipeline time: 2.2s, rows=540


Computing lift: 100%|██████████| 5000/5000 [00:11<00:00, 424.35it/s]


Validation (sample 1) summary: {'precision@N': 0.6925925925925924, 'recall@N': 0.019755921937839968, 'hit_rate': 1.0, 'coverage_fraction': 1.0}

--- Sample 2/10 (size=100) ---


...

Validation (sample 9) summary: {'precision@N': 0.688, 'recall@N': 0.015930563569878493, 'hit_rate': 0.98, 'coverage_fraction': 1.0}

--- Sample 10/10 (size=100) ---


Computing lift: 100%|██████████| 100/100 [00:01<00:00, 85.62it/s]


Complement pipeline time: 1.9s, rows=430


Computing lift: 100%|██████████| 5000/5000 [00:12<00:00, 394.54it/s]

Validation (sample 10) summary: {'precision@N': 0.758139534883721, 'recall@N': 0.01540516012005042, 'hit_rate': 1.0, 'coverage_fraction': 1.0}

Temporal metrics across samples (mean ± std):
       precision@N  recall@N  hit_rate  coverage_fraction
mean     0.715001  0.015105  0.987243                1.0
std      0.033965  0.001949  0.011205                0.0

        COMPLEMENTS PIPELINE – SUMMARY         

--- Runtime Performance ---
Avg complements pipeline time:    2.11s  (std=0.21)

--- Temporal Metrics (Mean ± Std) ---
precision@N         : 0.7150 ± 0.0340
recall@N            : 0.0151 ± 0.0019
hit_rate            : 0.9872 ± 0.0112
coverage_fraction   : 1.0000 ± 0.0000

--- Complement Impact Summary (Top 10 Risky Products) ---
    removed_product  avg_total_impact  avg_neighbor_CII_before  \
0               790               0.0                      0.0   
44            36680               0.0                      0.0   
32            24956               0.0                      0